# Milestone 6 Lab, notebook 7 of 8: The improver

## Where we are

| Notebook | What you do |
|---|---|
| 1. Late and corrected files, by hand | Work out by hand what three late files should do to the figures |
| 2. Corrections and incremental runs | Load the late files, recalculate only what they touch, and prove it equals a full rebuild |
| 3. The review desk | Decide the findings that need review, and watch the next run obey |
| 4. Run health and restatements | Read a run's health, and ask the analyst why a reported figure changed |
| 5. A judge with a reference answer | Test a judge on explanations whose grade you know, then use it |
| 6. The end-to-end evaluation | Score the whole system from files to answers, on two question sets |
| **7. The improver** (you are here) | An agent proposes a change to the analyst, you approve, and code keeps it or rolls it back |
| 8. The whole system | Run everything from files to answer, and look back over six milestones |

Model requests in this notebook: **about 120**. A run of this lab with its default small sets costs about 10 US cents.

## What this notebook is for

In milestones 3 and 4 the analyst had weaknesses, and a person fixed them: read the failures, find the cause, change the instructions, measure again. That is a loop, and a loop can be handed to an agent.

Here you build an **improver**: an AI agent that reads the analyst's results and proposes a change to the analyst. Three things keep it under control:

- **You approve** every proposal before it is saved.
- **Code decides** whether the change is kept, from held-out scores the improver never sees.
- **A budget** stops it.

At the end the improver proposes an improvement to its own instructions.

> This notebook needs `OPENROUTER_API_KEY`. With the default small sets it makes about 60 requests and costs about 7 US cents. With the full sets, about 15 cents.

In [ ]:
import pandas as pd
from lab_helpers import LAB_E2E, LAB_VERSIONS, show_doc, show_proposal

from retail_profit_stack.e2e import build_everything, evaluate, summary
from retail_profit_stack.evaluation import load_set
from retail_profit_stack.judge import make_judge

workspace = build_everything(LAB_E2E)
judge = make_judge()

## Step 1: The design

In [ ]:
show_doc("HLD.md", "10. The improver")

## Step 2: The business's settings

These numbers belong to the business. They live in code, where the improver can read them and cannot change them.

In [ ]:
from retail_profit_stack.improver import ALLOWED_MODELS, BUDGET, NOISE, OBJECTIVES, TOLERANCE

print("Objectives (a floor, on the held-out set):", OBJECTIVES)
print("Noise (must improve by more than this):   ", NOISE)
print("Tolerance (may not get worse by more):    ", TOLERANCE)
print("Budget for evaluations, in US dollars:    ", BUDGET)
print("Models the analyst may use:               ", ALLOWED_MODELS)

Accuracy and grade are compared as a difference: accuracy must rise by more than 0.1. Cost and time are compared as a share of the old value: 0.15 means 15%.

Time has a wide margin of 0.5, which is 50%. That comes straight from notebook 6, where the same analyst's time doubled between two runs. In the first trial of this lab the margin was 15%, and the code **kept** a change for being "faster" when nothing had got faster: one earlier run had simply stalled. A setting like this has to come from measuring.

The rule that uses them is short enough to read.

In [ ]:
import inspect

from retail_profit_stack.improver import verdict

print(inspect.getsource(verdict))

Try it on made-up scores before any model is involved.

In [ ]:
old = {"accuracy": 0.9, "grade": 2.5, "cost_per_question": 0.002, "seconds_per_question": 6.0}

print(verdict({**old, "accuracy": 0.95}, old))
print(verdict({**old, "accuracy": 1.0, "grade": 3.0}, old))
print(verdict({**old, "cost_per_question": 0.0015, "accuracy": 0.75}, old))
print(verdict({**old, "cost_per_question": 0.001, "accuracy": 0.6}, old))

Four cases: an improvement inside the noise is rolled back; a real one is kept; cheaper and clearly less accurate is rolled back; and anything below the business's floor is rolled back, however cheap.

## Step 3: A session, and version 1's scores

A **session** holds everything one round of improvement needs: the versions of the analyst, the scoreboard, and what has been spent. We give it a function that evaluates one version on one set. That function is notebook 6.

In [ ]:
from retail_profit_stack.analyst_versions import version_one
from retail_profit_stack.improver import Session

TINY = True


def evaluate_version(config, set_name):
    """Run one version of the analyst on one question set. Returns its scores and its answers."""
    table = evaluate(load_set(set_name, tiny=TINY), workspace, config=config, judge=judge, at_once=6)
    return summary(table), table


session = Session(evaluate=evaluate_version, versions=[version_one()])

Before anything is changed, measure version 1: once on the evaluation set, and twice on the held-out set.

In [ ]:
session.measure(1, "evaluation")
session.measure(1, "held-out")
session.measure(1, "held-out")

print("Spent so far:", round(session.spent, 4), "of", session.budget)
pd.DataFrame(session.scoreboard)

## Step 4: The improver's tools

The improver has four tools. It reads and proposes. Code does everything else.

| Tool | What it does |
|---|---|
| `read_results` | The current version, its scores, and its **mistakes on the evaluation set** |
| `propose_change` | A new version of the analyst's instructions and model. **Pauses for you** |
| `run_evaluation` | Code runs the new version and keeps it or rolls it back |
| `propose_recipe` | A new version of the improver's own recipe. **Pauses for you** |

Here is what `read_results` shows it. Look for what is **not** there.

In [ ]:
import json

from retail_profit_stack.improver import make_tools

tools_by_name = {tool.name: tool for tool in make_tools(session)}
report = json.loads(tools_by_name["read_results"].invoke({}))

print("It sees:", list(report))
print()
print("Mistakes on the evaluation set:", len(report["mistakes_on_evaluation_set"]))
for mistake in report["mistakes_on_evaluation_set"]:
    print("-", mistake["question"][:80], "|", mistake["mistake"], mistake["failed_checks"])

From the held-out set it sees **only the scores**. Not the questions, not the answers. That is the defence against overfitting, enforced by what the tool returns.

If the list of mistakes is empty, the analyst got every evaluation question right this time. The improver's recipe covers that case too.

## Step 5: The recipe

The improver's instructions are its **recipe**.

In [ ]:
print(session.recipes[-1])

Step 2 matters most: *one general rule*, and *no rule about one named product or question*. A rule about one question is overfitting written down.

## Step 6: The improver

The improver is a LangChain agent, like the analyst. In milestone 5 you wrote `interrupt()` by hand to make a workflow wait. LangChain's `HumanInTheLoopMiddleware` does the same for chosen tools: when the agent calls `propose_change` or `propose_recipe`, the run pauses until a person approves or rejects.

In [ ]:
from retail_profit_stack.agent import make_model
from retail_profit_stack.improver import make_improver, waiting_proposal

improver = make_improver(session, make_model(timeout=300))
round_1 = {"configurable": {"thread_id": "round-1"}, "recursion_limit": 40}

## Step 7: Round 1. The improver proposes

It reads the results and proposes a change. Then it stops.

In [ ]:
result = improver.invoke({"messages": [{"role": "user", "content": "Please carry out one round of your recipe."}]}, round_1)

show_proposal(waiting_proposal(result), session)

**Read the proposal.** Lines starting with `-` are removed from the analyst's instructions, and lines starting with `+` are added. This is your job in the loop:

- Is the reason sensible, given the mistakes you saw in step 4?
- Is the new rule **general**, or is it about one question?
- Does it remove anything that matters?

Nothing has been saved yet. To refuse, you would resume with `{"type": "reject", "message": "your reason"}`, and the improver would read your reason and try something else.

## Step 8: You approve, and code decides

If you are happy with it, run the next cell. The improver then calls `run_evaluation`. Code runs the new version once on the evaluation set and twice on the held-out set, and applies the rule from step 2. This takes about a minute.

In [ ]:
from langgraph.types import Command

APPROVE = Command(resume={"decisions": [{"type": "approve"}]})

result = improver.invoke(APPROVE, round_1)

pd.DataFrame(session.scoreboard)

In [ ]:
last_round = session.rounds[-1]

print("Version tried:", last_round["version"])
print("Verdict:      ", last_round["verdict"])
print("Version in use now:", session.current)
print()
pd.DataFrame([last_round["held_out_before"], last_round["held_out_after"]], index=["before", "after"])

Read the verdict. **Either outcome is a correct result.**

- **Kept** means a held-out score improved by more than the noise, and nothing got clearly worse.
- **Rolled back** means the change did not show a clear gain on questions the improver never saw. The analyst is exactly as it was.

When this lab was built, version 1 was already right on almost every question, so there was little room to improve. In one trial the improver found two answers that wrote amounts without listing them, and proposed one general rule. In another there were no mistakes, so it shortened the instructions. Neither moved a held-out score by more than chance. A rollback is the system working: an unproven change does not get in.

Notice who decided. The improver wrote the change, so it was not allowed to judge it. You approved it as reasonable. Code decided whether it worked.

## Step 9: Improving the improver

The improver has reached the last step of its recipe. It has seen what its change did, and now it proposes a change to **its own recipe**.

In [ ]:
show_proposal(waiting_proposal(result), session)

In [ ]:
result = improver.invoke(APPROVE, round_1)

print(result["messages"][-1].content)

A new improver, started from the new recipe, would run the next round. Each round can improve the recipe as well as the analyst. That is self-improvement on a small scale, and every step of it passed through your approval.

In [ ]:
print("Recipes so far:", len(session.recipes))
print("Spent:", round(session.spent, 4), "of", session.budget, "US dollars")

## Step 10: Into production

If a version was kept, it is saved as the analyst's current version, in the lab's own versions file. Rolling back later is one call: `set_current(1)`.

In [ ]:
from retail_profit_stack.analyst_versions import current_version, save_version, set_current

if LAB_VERSIONS.exists():
    LAB_VERSIONS.unlink()

if session.current != 1:
    kept = session.version(session.current)
    saved = save_version(kept["instructions"], kept["model"], kept["reason"], LAB_VERSIONS)
    set_current(saved["version"], LAB_VERSIONS)

print("The analyst now uses version", current_version(LAB_VERSIONS)["version"])

## Step 11: The budget

`run_evaluation` checks the budget before it spends. Set a tiny one and see.

In [ ]:
broke = Session(evaluate=evaluate_version, versions=[version_one(), {**version_one(), "version": 2}], budget=0.01)
broke.spent = 0.02

print({tool.name: tool for tool in make_tools(broke)}["run_evaluation"].invoke({}))

A budget in code is a good guard. A bug in that code could still spend money, so there is a second guard outside it: OpenRouter lets you put a **credit limit** on your key, at [openrouter.ai/settings/keys](https://openrouter.ai/settings/keys). It is enforced whatever your code does.

## What we now know

- An **improver** is an agent that reads another agent's mistakes and proposes a change to it.
- It may only **propose**. A person approves, and **code** keeps or rolls back.
- Only **held-out** scores decide, and the improver never sees those questions. That is the defence against overfitting.
- A change must beat the **noise**. Our first margin for time was too small, and it let a lucky run through.
- A **rollback** is a correct result. An unproven change does not get in.
- The business's objectives, budget, and allowed models are in code the improver cannot change.

**Next:** notebook 8, the last one. The whole system, from files to answer, and a look back over six milestones.